# 전체 데이터 대상 테스트

In [2]:
import os
import glob
import json
import pandas as pd
import numpy as np
import time
import warnings
from tqdm import tqdm
from konlpy.tag import Mecab
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

# [추가] 불필요한 경고 메시지 끄기
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn.feature_extraction.text")

# =========================================================
# [설정] 실험 조건 설정
# =========================================================
DATA_DIRS = ["data/대법원_판례", "data/하위법원_판례"] 
TARGET_STOPWORD_DIRS = ["stopword_공통", "stopword_판례", "stopword_법령"]
TOP_N_CLASSES = 10
NUM_ITERATIONS = 5 

mecab = Mecab()

def pre_tokenize(text):
    """명사만 추출하여 공백으로 구분된 문자열 생성 (Pre-tokenization)"""
    if not text: return ""
    return " ".join([word for word, pos in mecab.pos(text) if pos.startswith('N')])

def load_and_preprocess_data(data_dirs_list, top_n=None):
    all_files = []
    for d in data_dirs_list:
        files = glob.glob(os.path.join(d, "**", "*.json"), recursive=True)
        all_files.extend(files)
    
    texts = []
    labels = []
    
    # 1. 데이터 로딩
    for file_path in tqdm(all_files, desc="JSON Loading"):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                content = data.get('판례내용', "").replace("<br/>", " ").replace("\n", " ")
                label = data.get('사건명', "")
                if content and label:
                    texts.append(content)
                    labels.append(label)
        except: continue
    
    df = pd.DataFrame({'text': texts, 'label': labels})
    
    # 2. Top N 필터링
    if top_n:
        target_classes = df['label'].value_counts().head(top_n).index.tolist()
        df = df[df['label'].isin(target_classes)].copy()
    
    # 3. [핵심] 사전 토큰화 (전체 데이터에 대해 1회만 수행)
    tqdm.pandas(desc="Pre-tokenizing (Mecab)")
    df['tokenized_text'] = df['text'].progress_apply(pre_tokenize)
    
    return train_test_split(
        df['tokenized_text'], df['label'], 
        test_size=0.2, random_state=42, stratify=df['label']
    )

print(f">>> 1. 데이터셋 로딩 및 사전 토큰화 시작...")
X_train, X_test, y_train, y_test = load_and_preprocess_data(DATA_DIRS, TOP_N_CLASSES)

# =========================================================
# [Part 2] 불용어 리스트 복구 (동일)
# =========================================================
def load_stopword_flexible(folder, suffix):
    prefixes = ["stopwords_exp_", "candidates_"]
    for pre in prefixes:
        path = os.path.join(folder, f"{pre}{suffix}")
        if os.path.exists(path):
            try: return set(pd.read_csv(path)['word'].astype(str).tolist())
            except: continue
    return None

GENERAL_STOPWORD_FILE = "stopwords.txt"
if not os.path.exists(GENERAL_STOPWORD_FILE):
    with open(GENERAL_STOPWORD_FILE, "w", encoding="utf-8") as f:
        f.write("이\n그\n저\n것\n수\n등\n들\n및\n에\n의\n를\n가\n은\n는")

set_general = set([line.strip() for line in open(GENERAL_STOPWORD_FILE, 'r', encoding='utf-8') if line.strip()])

# =========================================================
# [Part 3] 실험 시나리오 자동 생성
# =========================================================
stopword_sets = {"1. Baseline (None)": set(), "2. General Only": set_general}
for folder in TARGET_STOPWORD_DIRS:
    if not os.path.exists(folder): continue
    tag = f"[{folder.replace('stopword_', '')}]"
    s_a, s_b, s_c, s_e = [load_stopword_flexible(folder, s) for s in ["A_zipf.csv", "B_entropy.csv", "C_mpvp.csv", "E_intersection.csv"]]
    if s_a: stopword_sets[f"{tag} Exp A (Zipf)"] = s_a
    if s_b: stopword_sets[f"{tag} Exp B (Ent)"] = s_b
    if s_c: stopword_sets[f"{tag} Exp C (MPVP)"] = s_c
    if s_e: 
        stopword_sets[f"{tag} Exp E (Intersect)"] = s_e
        stopword_sets[f"{tag} Hybrid (Gen+E)"] = set_general.union(s_e)

# =========================================================
# [Part 4] 대규모 벤치마크 실행 (Pre-tokenized 데이터 사용)
# =========================================================
print(f"\n>>> 3. [확장 실험] 판례 분류 성능 비교 (각 {NUM_ITERATIONS}회 반복)")
results = []
print("-" * 105)
print(f"{'Experiment':<35} | {'Acc':<8} | {'F1':<8} | {'Avg Time':<10} | {'Count'}")
print("-" * 105)

for name, stop_set in stopword_sets.items():
    stop_list = list(stop_set) if stop_set else None
    times, accuracies, f1s = [], [], []
    
    for _ in range(NUM_ITERATIONS):
        # [수정] 이미 토큰화된 텍스트를 사용하므로 tokenizer=None, token_pattern 설정
        pipeline = Pipeline([
            ('vect', TfidfVectorizer(
                stop_words=stop_list, 
                max_features=5000,
                tokenizer=None, 
                token_pattern=r"(?u)\b\w\w+\b" # 한 글자 단어 제외 기본 패턴
            )),
            ('clf', LogisticRegression(C=5.0, max_iter=1000, n_jobs=-1)) 
        ])
        
        start_time = time.time()
        try:
            pipeline.fit(X_train, y_train)
            pred = pipeline.predict(X_test)
            times.append(time.time() - start_time)
            accuracies.append(accuracy_score(y_test, pred))
            f1s.append(f1_score(y_test, pred, average='weighted'))
        except Exception as e:
            print(f"Error in {name}: {e}")
            break
            
    if times:
        avg_acc, avg_f1, avg_time = np.mean(accuracies), np.mean(f1s), np.mean(times)
        print(f"{name:<35} | {avg_acc:.4f}   | {avg_f1:.4f}   | {avg_time:.3f}s (±{np.std(times):.2f}) | {len(stop_set)}")
        results.append({'Experiment': name, 'Count': len(stop_set), 'Accuracy': avg_acc, 'F1_Score': avg_f1, 'Avg_Time': avg_time})

# [Part 5] 결과 저장 및 정렬 보완
if results:
    # 1. 정렬 기준 강화: 정확도(높은순) -> F1(높은순) -> 시간(짧은순)
    final_df = pd.DataFrame(results).sort_values(
        by=['Accuracy', 'F1_Score', 'Avg_Time'], 
        ascending=[False, False, True]
    ).round(4)
    
    # 2. 순위(Rank) 컬럼 명시적 추가 (선택 사항)
    final_df.insert(0, 'Rank', range(1, len(final_df) + 1))
    
    print("\n" + "="*50)
    print(f" [최종 종합 순위 (Top {TOP_N_CLASSES} 클래스, {NUM_ITERATIONS}회 평균)]")
    print("="*50)
    print(final_df.to_string(index=False)) # 인덱스 없이 깔끔하게 출력
    
    # 3. 결과 저장
    output_name = f"benchmark_results_precedents_top{TOP_N_CLASSES}.csv"
    final_df.to_csv(output_name, index=False, encoding='utf-8-sig')
    print(f"\n>>> 결과가 '{output_name}'로 저장되었습니다.")

>>> 1. 데이터셋 로딩 및 사전 토큰화 시작...


Pre-tokenizing (Mecab): 100%|██████████| 4763/4763 [00:18<00:00, 260.24it/s]



>>> 3. [확장 실험] 판례 분류 성능 비교 (각 5회 반복)
---------------------------------------------------------------------------------------------------------
Experiment                          | Acc      | F1       | Avg Time   | Count
---------------------------------------------------------------------------------------------------------
1. Baseline (None)                  | 0.8730   | 0.8708   | 3.636s (±0.66) | 0
2. General Only                     | 0.8762   | 0.8741   | 3.524s (±0.05) | 600
[공통] Exp A (Zipf)                   | 0.8636   | 0.8612   | 3.159s (±0.02) | 167
[공통] Exp B (Ent)                    | 0.8657   | 0.8632   | 3.047s (±0.04) | 226
[공통] Exp C (MPVP)                   | 0.8720   | 0.8699   | 3.481s (±0.04) | 54
[공통] Exp E (Intersect)              | 0.8709   | 0.8687   | 3.423s (±0.03) | 44
[공통] Hybrid (Gen+E)                 | 0.8709   | 0.8690   | 3.490s (±0.02) | 639
[판례] Exp A (Zipf)                   | 0.8604   | 0.8583   | 3.154s (±0.04) | 132
[판례] Exp B (Ent)           

# 법령 대상 테스트

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import time
import unicodedata
import warnings
from tqdm import tqdm
from konlpy.tag import Mecab
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

# [추가] 경고 메시지 무시
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn.feature_extraction.text")

# =========================================================
# [설정] 데이터 경로 및 실험 옵션
# =========================================================
TEXT_DIR = "data/법률_정제_완료_txt화" 
META_FILE = "data/전체_법률_목록(2025-12-01).csv"
TARGET_STOPWORD_DIRS = ["stopword_공통", "stopword_법령", "stopword_판례"]
TOP_N_CLASSES = 10
NUM_ITERATIONS = 5

mecab = Mecab()

def pre_tokenize(text):
    """명사만 추출하여 공백으로 구분된 문자열 생성"""
    if not text: return ""
    return " ".join([word for word, pos in mecab.pos(text) if pos.startswith('N')])

# =========================================================
# [Part 1] 데이터셋 구축 (Pre-tokenization 적용)
# =========================================================
print(f">>> 1. 법령 데이터셋 구축 및 사전 토큰화 시작...")

def load_statute_data(text_dir, meta_file, top_n=10):
    # 메타데이터 로드
    try:
        try: meta_df = pd.read_csv(meta_file, encoding='cp949')
        except: meta_df = pd.read_csv(meta_file, encoding='utf-8')
    except Exception as e:
        print(f"[Error] 메타파일 로드 실패: {e}"); return None, None, None, None

    label_map = dict(zip(meta_df['법령명'].str.strip(), meta_df['법령분야명'].str.strip()))
    
    texts, labels = [], []
    txt_files = glob.glob(os.path.join(text_dir, "**", "*.txt"), recursive=True)
    
    # 1. 파일 매칭 및 로딩
    for file_path in tqdm(txt_files, desc="Matching & Loading"):
        filename_raw = os.path.basename(file_path).replace(".txt", "").strip()
        filename = unicodedata.normalize('NFC', filename_raw) # Mac 호환성 유지
        
        if filename in label_map:
            label = label_map[filename]
            if pd.isna(label) or not str(label).strip(): continue
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    texts.append(f.read())
                    labels.append(label)
            except: continue
    
    df = pd.DataFrame({'text': texts, 'label': labels})
    if len(df) == 0: return [], [], [], []

    # 2. Top N 필터링
    if top_n:
        target_classes = df['label'].value_counts().head(top_n).index.tolist()
        df = df[df['label'].isin(target_classes)].copy()
    
    # 3. [핵심] 사전 토큰화 (1회 수행)
    tqdm.pandas(desc="Pre-tokenizing (Mecab)")
    df['tokenized_text'] = df['text'].progress_apply(pre_tokenize)
    
    print(f"   - 최종 학습 데이터: {len(df)}건")
    return train_test_split(df['tokenized_text'], df['label'], test_size=0.2, random_state=42, stratify=df['label'])

try:
    X_train, X_test, y_train, y_test = load_statute_data(TEXT_DIR, META_FILE, TOP_N_CLASSES)
    if len(X_train) == 0: raise ValueError("Empty Data")
except Exception as e:
    print(f"[Warning] 데이터 로드 실패({e}). 더미 데이터를 사용합니다.")
    X_train, y_train = ["제1조 목적 법령"]*50, ["행정"]*50
    X_test, y_test = ["제1조 목적 법령"]*10, ["행정"]*10

# =========================================================
# [Part 2] 불용어 리스트 복구 (동일)
# =========================================================
def load_stopword_flexible(folder, suffix):
    prefixes = ["stopwords_exp_", "candidates_"]
    for pre in prefixes:
        path = os.path.join(folder, f"{pre}{suffix}")
        if os.path.exists(path):
            try: return set(pd.read_csv(path)['word'].astype(str).tolist())
            except: continue
    return None

GENERAL_STOPWORD_FILE = "stopwords.txt"
if not os.path.exists(GENERAL_STOPWORD_FILE):
    with open(GENERAL_STOPWORD_FILE, "w", encoding="utf-8") as f:
        f.write("이\n그\n저\n것\n수\n등\n들\n및\n에\n의\n를\n가\n은\n는")

set_general = set([line.strip() for line in open(GENERAL_STOPWORD_FILE, 'r', encoding='utf-8') if line.strip()])

# =========================================================
# [Part 3] 실험 시나리오 자동 생성
# =========================================================
stopword_sets = {"1. Baseline (None)": set(), "2. General Only": set_general}
for folder in TARGET_STOPWORD_DIRS:
    if not os.path.exists(folder): continue
    tag = f"[{folder.replace('stopword_', '')}]"
    s_a, s_b, s_c, s_e = [load_stopword_flexible(folder, s) for s in ["A_zipf.csv", "B_entropy.csv", "C_mpvp.csv", "E_intersection.csv"]]
    if s_a: stopword_sets[f"{tag} Exp A (Zipf)"] = s_a
    if s_b: stopword_sets[f"{tag} Exp B (Ent)"] = s_b
    if s_c: stopword_sets[f"{tag} Exp C (MPVP)"] = s_c
    if s_e: 
        stopword_sets[f"{tag} Exp E (Intersect)"] = s_e
        stopword_sets[f"{tag} Hybrid (Gen+E)"] = set_general.union(s_e)

# =========================================================
# [Part 4] 반복 벤치마크 실행 (Optimized)
# =========================================================
print(f"\n>>> 3. [법령 분류] 성능 비교 (각 {NUM_ITERATIONS}회 반복)")

results = []
print("-" * 105)
print(f"{'Experiment':<35} | {'Acc':<8} | {'F1':<8} | {'Avg Time':<10} | {'Count'}")
print("-" * 105)

for name, stop_set in stopword_sets.items():
    stop_list = list(stop_set) if stop_set else None
    times, accuracies, f1s = [], [], []
    
    for _ in range(NUM_ITERATIONS):
        # 이미 토큰화된 텍스트를 공백 기준으로 처리
        pipeline = Pipeline([
            ('vect', TfidfVectorizer(
                stop_words=stop_list, 
                max_features=5000,
                tokenizer=None, 
                token_pattern=r"(?u)\b\w\w+\b" # 2글자 이상 단어만 추출
            )),
            ('clf', LogisticRegression(C=5.0, max_iter=1000, n_jobs=-1)) 
        ])
        
        start_time = time.time()
        try:
            pipeline.fit(X_train, y_train)
            pred = pipeline.predict(X_test)
            times.append(time.time() - start_time)
            accuracies.append(accuracy_score(y_test, pred))
            f1s.append(f1_score(y_test, pred, average='weighted'))
        except Exception as e:
            print(f"Error in {name}: {e}"); break
    
    if times:
        avg_acc, avg_f1, avg_time = np.mean(accuracies), np.mean(f1s), np.mean(times)
        print(f"{name:<35} | {avg_acc:.4f}   | {avg_f1:.4f}   | {avg_time:.3f}s (±{np.std(times):.2f}) | {len(stop_set)}")
        results.append({
            'Experiment': name, 'Count': len(stop_set), 
            'Accuracy': avg_acc, 'F1_Score': avg_f1, 
            'Avg_Time': avg_time, 'Time_Std_Dev': np.std(times)
        })

# [Part 5] 결과 저장
if results:
    final_df = pd.DataFrame(results).sort_values(by='Accuracy', ascending=False).round(4)
    print("\n[최종 종합 순위 (5회 평균)]\n", final_df)
    final_df.to_csv(f"benchmark_results_statute_top{TOP_N_CLASSES}.csv", index=False, encoding='utf-8-sig')

>>> 1. 법령 데이터셋 구축 및 사전 토큰화 시작...


Pre-tokenizing (Mecab): 100%|██████████| 486/486 [00:05<00:00, 90.99it/s] 


   - 최종 학습 데이터: 486건

>>> 3. [법령 분류] 성능 비교 (각 5회 반복)
---------------------------------------------------------------------------------------------------------
Experiment                          | Acc      | F1       | Avg Time   | Count
---------------------------------------------------------------------------------------------------------


NameError: name 'stopword_sets' is not defined

# 전체 판례 대상 테스트

In [6]:
import os
import glob
import json
import pandas as pd
import numpy as np
import time
import warnings
from tqdm import tqdm
from konlpy.tag import Mecab
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

# [추가] 불필요한 경고 메시지 끄기
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn.feature_extraction.text")

# =========================================================
# [설정] 실험 조건 설정
# =========================================================
DATA_DIRS = ["data/대법원_판례", "data/하위법원_판례"] 
TARGET_STOPWORD_DIRS = ["stopword_공통", "stopword_판례", "stopword_법령"]
TOP_N_CLASSES = 10
NUM_ITERATIONS = 5 

mecab = Mecab()

def pre_tokenize(text):
    """명사만 추출하여 공백으로 구분된 문자열 생성 (Pre-tokenization)"""
    if not text: return ""
    return " ".join([word for word, pos in mecab.pos(text) if pos.startswith('N')])

def load_and_preprocess_data(data_dirs_list, top_n=None):
    all_files = []
    for d in data_dirs_list:
        files = glob.glob(os.path.join(d, "**", "*.json"), recursive=True)
        all_files.extend(files)
    
    texts = []
    labels = []
    
    # 1. 데이터 로딩
    for file_path in tqdm(all_files, desc="JSON Loading"):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                content = data.get('판례내용', "").replace("<br/>", " ").replace("\n", " ")
                label = data.get('사건명', "")
                if content and label:
                    texts.append(content)
                    labels.append(label)
        except: continue
    
    df = pd.DataFrame({'text': texts, 'label': labels})
    
    # 2. Top N 필터링
    if top_n:
        target_classes = df['label'].value_counts().head(top_n).index.tolist()
        df = df[df['label'].isin(target_classes)].copy()
    
    # 3. [핵심] 사전 토큰화 (전체 데이터에 대해 1회만 수행)
    tqdm.pandas(desc="Pre-tokenizing (Mecab)")
    df['tokenized_text'] = df['text'].progress_apply(pre_tokenize)
    
    return train_test_split(
        df['tokenized_text'], df['label'], 
        test_size=0.2, random_state=42, stratify=df['label']
    )

print(f">>> 1. 데이터셋 로딩 및 사전 토큰화 시작...")
X_train, X_test, y_train, y_test = load_and_preprocess_data(DATA_DIRS, TOP_N_CLASSES)

# =========================================================
# [Part 2] 불용어 리스트 복구 (동일)
# =========================================================
def load_stopword_flexible(folder, suffix):
    prefixes = ["stopwords_exp_", "candidates_"]
    for pre in prefixes:
        path = os.path.join(folder, f"{pre}{suffix}")
        if os.path.exists(path):
            try: return set(pd.read_csv(path)['word'].astype(str).tolist())
            except: continue
    return None

GENERAL_STOPWORD_FILE = "stopwords.txt"
if not os.path.exists(GENERAL_STOPWORD_FILE):
    with open(GENERAL_STOPWORD_FILE, "w", encoding="utf-8") as f:
        f.write("이\n그\n저\n것\n수\n등\n들\n및\n에\n의\n를\n가\n은\n는")

set_general = set([line.strip() for line in open(GENERAL_STOPWORD_FILE, 'r', encoding='utf-8') if line.strip()])

# =========================================================
# [Part 3] 실험 시나리오 필터링 (판례 데이터 전용으로 제한)
# =========================================================
# 기본 세트 설정
stopword_sets = {
    "1. Baseline (None)": set(), 
    "2. General Only": set_general
}

# [수정] 오직 'stopword_판례' 폴더만 타겟팅
TARGET_FOLDER = "stopword_판례_통합"

if os.path.exists(TARGET_FOLDER):
    tag = "[Precedent-Specific]"
    # 각 실험 알고리즘별 불용어 로드
    s_a = load_stopword_flexible(TARGET_FOLDER, "A_zipf.csv")
    s_b = load_stopword_flexible(TARGET_FOLDER, "B_entropy.csv")
    s_c = load_stopword_flexible(TARGET_FOLDER, "C_mpvp.csv")
    s_e = load_stopword_flexible(TARGET_FOLDER, "E_intersection.csv")
    
    if s_a: stopword_sets[f"{tag} Exp A (Zipf)"] = s_a
    if s_b: stopword_sets[f"{tag} Exp B (Ent)"] = s_b
    if s_c: stopword_sets[f"{tag} Exp C (MPVP)"] = s_c
    if s_e: 
        stopword_sets[f"{tag} Exp E (Intersect)"] = s_e
        # 하이브리드: 일반 불용어 + 판례 전용 교집합 불용어
        stopword_sets[f"{tag} Hybrid (Gen+E)"] = set_general.union(s_e)

print(f">>> 생성된 실험 시나리오: {list(stopword_sets.keys())}")

# =========================================================
# [Part 4] 대규모 벤치마크 실행 (Pre-tokenized 데이터 사용)
# =========================================================
print(f"\n>>> 3. [확장 실험] 판례 분류 성능 비교 (각 {NUM_ITERATIONS}회 반복)")
results = []
print("-" * 105)
print(f"{'Experiment':<35} | {'Acc':<8} | {'F1':<8} | {'Avg Time':<10} | {'Count'}")
print("-" * 105)

for name, stop_set in stopword_sets.items():
    stop_list = list(stop_set) if stop_set else None
    times, accuracies, f1s = [], [], []
    
    for _ in range(NUM_ITERATIONS):
        # [수정] 이미 토큰화된 텍스트를 사용하므로 tokenizer=None, token_pattern 설정
        pipeline = Pipeline([
            ('vect', TfidfVectorizer(
                stop_words=stop_list, 
                max_features=5000,
                tokenizer=None, 
                token_pattern=r"(?u)\b\w\w+\b" # 한 글자 단어 제외 기본 패턴
            )),
            ('clf', LogisticRegression(C=5.0, max_iter=1000, n_jobs=-1)) 
        ])
        
        start_time = time.time()
        try:
            pipeline.fit(X_train, y_train)
            pred = pipeline.predict(X_test)
            times.append(time.time() - start_time)
            accuracies.append(accuracy_score(y_test, pred))
            f1s.append(f1_score(y_test, pred, average='weighted'))
        except Exception as e:
            print(f"Error in {name}: {e}")
            break
            
    if times:
        avg_acc, avg_f1, avg_time = np.mean(accuracies), np.mean(f1s), np.mean(times)
        print(f"{name:<35} | {avg_acc:.4f}   | {avg_f1:.4f}   | {avg_time:.3f}s (±{np.std(times):.2f}) | {len(stop_set)}")
        results.append({'Experiment': name, 'Count': len(stop_set), 'Accuracy': avg_acc, 'F1_Score': avg_f1, 'Avg_Time': avg_time})

# [Part 5] 결과 저장 (정확도 -> F1 -> 시간 순으로 정렬)
if results:
    final_df = pd.DataFrame(results).sort_values(
        by=['Accuracy', 'F1_Score', 'Avg_Time'], 
        ascending=[False, False, True]
    ).round(4)
    
    # 순위 컬럼 추가
    final_df.insert(0, 'Rank', range(1, len(final_df) + 1))
    
    print("\n" + "="*60)
    print(f" [판례 전용 불용어 성능 순위 (Top {TOP_N_CLASSES})]")
    print("="*60)
    print(final_df.to_string(index=False))
    
    final_df.to_csv(f"benchmark_precedent_only_top{TOP_N_CLASSES}.csv", index=False, encoding='utf-8-sig')

>>> 1. 데이터셋 로딩 및 사전 토큰화 시작...


Pre-tokenizing (Mecab): 100%|██████████| 4763/4763 [00:20<00:00, 236.45it/s]


>>> 생성된 실험 시나리오: ['1. Baseline (None)', '2. General Only', '[Precedent-Specific] Exp A (Zipf)', '[Precedent-Specific] Exp B (Ent)', '[Precedent-Specific] Exp C (MPVP)', '[Precedent-Specific] Exp E (Intersect)', '[Precedent-Specific] Hybrid (Gen+E)']

>>> 3. [확장 실험] 판례 분류 성능 비교 (각 5회 반복)
---------------------------------------------------------------------------------------------------------
Experiment                          | Acc      | F1       | Avg Time   | Count
---------------------------------------------------------------------------------------------------------
1. Baseline (None)                  | 0.8730   | 0.8708   | 3.736s (±0.60) | 0
2. General Only                     | 0.8762   | 0.8741   | 3.557s (±0.05) | 600
[Precedent-Specific] Exp A (Zipf)   | 0.8604   | 0.8586   | 3.173s (±0.01) | 187
[Precedent-Specific] Exp B (Ent)    | 0.8395   | 0.8370   | 3.009s (±0.11) | 519
[Precedent-Specific] Exp C (MPVP)   | 0.8562   | 0.8544   | 3.179s (±0.06) | 221
[Precedent-Specifi